# Import All Libraries

In [77]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer,make_column_transformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor , GradientBoostingRegressor
from xgboost import XGBRegressor

In [78]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Loading Dataset

In [79]:
df = pd.read_csv('/content/drive/MyDrive/Housing.csv')

In [80]:
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


# Data Understanding & Data Cleaning

In [81]:
df.shape

(545, 13)

In [82]:
df.columns

Index(['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom', 'basement', 'hotwaterheating', 'airconditioning',
       'parking', 'prefarea', 'furnishingstatus'],
      dtype='object')

In [83]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [84]:
df.describe()

,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


In [85]:
# checking null values
df.isnull().sum()

,0
price,0
area,0
bedrooms,0
bathrooms,0
stories,0
mainroad,0
guestroom,0
basement,0
hotwaterheating,0
airconditioning,0


In [86]:
df.duplicated().sum()

np.int64(0)

# Data Cleaning

In [87]:
df.dropna()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [88]:
df.drop_duplicates()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [89]:
df['bedrooms'].unique()

array([4, 3, 5, 2, 6, 1])

In [90]:
df['bathrooms'].unique()

array([2, 4, 1, 3])

In [91]:
df['stories'].unique()

array([3, 4, 2, 1])

In [92]:
df['mainroad'].unique()

array(['yes', 'no'], dtype=object)

In [93]:
df['guestroom'].unique()

array(['no', 'yes'], dtype=object)

In [94]:
df['basement'].unique()

array(['no', 'yes'], dtype=object)

In [95]:
df['hotwaterheating'].unique()

array(['no', 'yes'], dtype=object)

In [96]:
df['airconditioning'].unique()

array(['yes', 'no'], dtype=object)

In [97]:
df['parking'].unique()

array([2, 3, 0, 1])

In [98]:
df['prefarea'].unique()

array(['yes', 'no'], dtype=object)

In [99]:
df['furnishingstatus'].unique()

array(['furnished', 'semi-furnished', 'unfurnished'], dtype=object)

In [100]:
df['price'].unique()

array([13300000, 12250000, 12215000, 11410000, 10850000, 10150000,
        9870000,  9800000,  9681000,  9310000,  9240000,  9100000,
        8960000,  8890000,  8855000,  8750000,  8680000,  8645000,
        8575000,  8540000,  8463000,  8400000,  8295000,  8190000,
        8120000,  8080940,  8043000,  7980000,  7962500,  7910000,
        7875000,  7840000,  7700000,  7560000,  7525000,  7490000,
        7455000,  7420000,  7350000,  7343000,  7245000,  7210000,
        7140000,  7070000,  7035000,  7000000,  6930000,  6895000,
        6860000,  6790000,  6755000,  6720000,  6685000,  6650000,
        6629000,  6615000,  6580000,  6510000,  6475000,  6440000,
        6419000,  6405000,  6300000,  6293000,  6265000,  6230000,
        6195000,  6160000,  6125000,  6107500,  6090000,  6083000,
        6020000,  5950000,  5943000,  5880000,  5873000,  5866000,
        5810000,  5803000,  5775000,  5740000,  5652500,  5600000,
        5565000,  5530000,  5523000,  5495000,  5460000,  5425

# Data Preprocessing

In [101]:
x = df.drop('price',axis=1)
y = df['price']

In [102]:
x

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...
540,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [103]:
y

,price
0,13300000
1,12250000
2,12250000
3,12215000
4,11410000
...,...
540,1820000
541,1767150
542,1750000
543,1750000


In [104]:
x.shape , y.shape

((545, 12), (545,))

# Column Transformer

In [105]:
df.columns

Index(['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom', 'basement', 'hotwaterheating', 'airconditioning',
       'parking', 'prefarea', 'furnishingstatus'],
      dtype='object')

In [106]:
num_cols = ['area','bedrooms','bathrooms','stories','parking']

In [107]:
binary_cols = ['mainroad','guestroom','basement','hotwaterheating','airconditioning','prefarea']

In [108]:
ordinal_cols = ['furnishingstatus']

In [109]:
preprocessor = ColumnTransformer([
    ('num',StandardScaler(),num_cols),
    ('binary',OneHotEncoder(drop='if_binary',handle_unknown='ignore'),binary_cols),
    ('furnish',OrdinalEncoder(categories=[['unfurnished','semi-furnished','furnished']]),ordinal_cols)
])

In [110]:
preprocessor

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['area', 'bedrooms', 'bathrooms', 'stories',
                                  'parking']),
                                ('binary',
                                 OneHotEncoder(drop='if_binary',
                                               handle_unknown='ignore'),
                                 ['mainroad', 'guestroom', 'basement',
                                  'hotwaterheating', 'airconditioning',
                                  'prefarea']),
                                ('furnish',
                                 OrdinalEncoder(categories=[['unfurnished',
                                                             'semi-furnished',
                                                             'furnished']]),
                                 ['furnishingstatus'])])

# Applying Train-Test-Split

In [111]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

x_train.shape, x_test.shape

((436, 12), (109, 12))

# Multi Model Deveplopment





In [112]:
models = {
    'LinearRegression' : LinearRegression(),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'KNN': KNeighborsRegressor(),
    'RandomForest': RandomForestRegressor(n_estimators=200,random_state=42),
    "XGBoost": XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        random_state=42,
        verbosity=0
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        min_samples_split=5,
        random_state=42)

}

# Model Evaluation & Comparative With Pipeline

In [113]:
results = {}

for name, model in models.items():
  pipe = Pipeline([
      ('preprocessig', preprocessor),
      ('model',model)
  ])

  pipe.fit(x_train,y_train)
  preds = pipe.predict(x_test)

  r2 = r2_score(y_test,preds)
  mae = mean_absolute_error(y_test,preds)
  rmse = np.sqrt(mean_squared_error(y_test,preds))

  results[name] = r2

  print("===========================")
  print(f"Model: {name}")
  print(f"R2 Score: {r2:.4f}")
  print(f"MAE: {mae:,.0f}")
  print(f"RMSE: {rmse:,.0f}")

Model: LinearRegression
R2 Score: 0.6495
MAE: 979,680
RMSE: 1,331,071
Model: DecisionTree
R2 Score: 0.4177
MAE: 1,265,651
RMSE: 1,715,615
Model: KNN
R2 Score: 0.5721
MAE: 1,080,846
RMSE: 1,470,708
Model: RandomForest
R2 Score: 0.6182
MAE: 1,011,168
RMSE: 1,389,249
Model: XGBoost
R2 Score: 0.6456
MAE: 995,440
RMSE: 1,338,339
Model: GradientBoosting
R2 Score: 0.6324
MAE: 1,004,122
RMSE: 1,363,196


# Best Model Selection & Perform

In [114]:
best_model_name = max(results,key= results.get)
print('Best Model is:',best_model_name)
print(f"Best R2: {results[best_model_name]:.4f}")

Best Model is: LinearRegression
Best R2: 0.6495


In [115]:
final_model = models[best_model_name]

final_pipeline = Pipeline([
    ('preprocessng', preprocessor),
    ('Model', final_model)
])

final_pipeline.fit(x_train,y_train)


Pipeline(steps=[('preprocessng',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['area', 'bedrooms',
                                                   'bathrooms', 'stories',
                                                   'parking']),
                                                 ('binary',
                                                  OneHotEncoder(drop='if_binary',
                                                                handle_unknown='ignore'),
                                                  ['mainroad', 'guestroom',
                                                   'basement',
                                                   'hotwaterheating',
                                                   'airconditioning',
                                                   'prefarea']),
                                                 ('furnish',
                                                  OrdinalEncoder(categories=[['unfurnished',
                                                                              'semi-furnished',
                                                                              'furnished']]),
                                                  ['furnishingstatus'])])),
                ('Model', LinearRegression())])

# Realtime Predictions

In [116]:
new_house = pd.DataFrame({
    'area': [5000],
    'bedrooms': [4],
    'bathrooms': [2],
    'stories': [2],
    'mainroad': ['yes'],
    'guestroom': ['yes'],
    'basement': ['yes'],
    'hotwaterheating': ['no'],
    'airconditioning': ['yes'],
    'parking': ['1'],
    'prefarea': ['yes'],
    'furnishingstatus': ['furnished']
})

predicted_price = final_pipeline.predict(new_house)
print(f"\n Predicted House Price: {predicted_price[0]:,.0f}")



 Predicted House Price: 7,427,645


In [118]:
import joblib
import os

joblib.dump(final_pipeline, "house_model.pkl")
print("✅ Model saved! Size:", os.path.getsize("house_model.pkl"), "bytes")

# Verify
test_load = joblib.load("house_model.pkl")
print("✅ Model verified:", type(test_load))

# Download
from google.colab import files
files.download("house_model.pkl")

✅ Model saved! Size: 5769 bytes
✅ Model verified: <class 'sklearn.pipeline.Pipeline'>


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [119]:
import sklearn
print(sklearn.__version__)

1.6.1
